
Bi orthogonality
https://www.cl.cam.ac.uk/~amp12/papers/steibt/steibt.pdf
"
it's just a way to present the usual logical relations argument
there's programs and their orthogonal "environments"
and each program is mapped to the set of environments that makes it terminate, and vice versa
and if you go round-trip, you get exactly {true, false} -> {programs that either are stuck, or reduce to true or false}
then you show that all those sets closed under the round-trip operation are a model of your type theory
"

Syntax

nbe like came from a lispy system right? https://epub.ub.uni-muenchen.de/4261/1/4261.pdf

https://en.wikipedia.org/wiki/Normalisation_by_evaluation
https://www.cse.chalmers.se/~abela/habil.pdf

In [ ]:
type Term = str
Bool0 : frozenset[Term] = {"True", "False"}
Unit0 : frozenset[Term] = {"()"}
Type0 = {"Bool", "Unit"}

type Code = str
def El(c : Code):
    assert c in Type
    return eval(c)


In [ ]:
def Bool(ctx, n):
    if n >= 0:
        return ctx + {"True", "False"}
    else:
        # and the other operators + appropriate operators in context
        return {x + " and " y for i in range(n) for x in Bool(i) for y in Bool(n - i)} 

ctx = list[tuple(Var, Type)]


`repr` is sometimes an inverse function to eval.

python 3.14 t strings might be useful
There's also pickling / cloudpickling


In [ ]:
assert eval(repr(True)) == True
eval(repr(()))

()

In [8]:
from dataclasses import dataclass
@dataclass(frozen=True)
class App:
    f : str
    args : tuple["App", ...]

eval(repr(App("a", ())))

App(f='a', args=())

In [12]:
eval(repr(frozenset([1,2])))

frozenset({1, 2})

In [10]:
repr(lambda x: x)
def foo(x):
    return x
repr(foo)

'<function foo at 0x7e69389377e0>'

In [ ]:

def reify(): ...
def reflect(): ...
def nbe(): ...
def is_eq(t, s, A): ... # def_eq?
def is_type(A): ...
def has_type(t, A): ...



Everything needs to be lifted to work symbolically. Unless you change your interpreter.
This is like rosette
metaocaml
partial evaluation


In [1]:
@dataclass(frozen=True)
class Code():
    code : str
    def __add__(self, other):
        if not isinstance(other, Code):
            other = quote(other)
        return Code(self.code + " + " + other.code)
    def __radd__(self, other):
        if not isinstance(other, Code):
            other = quote(other)
        return Code(other.code + " + " + self.code)
    def __mul__(self, other):
        if not isinstance(other, Code):
            other = quote(other)
        return Code(self.code + " * " + other.code)
    def __call__(self, *args):
        args = [arg if isinstance(arg, Code) else quote(arg) for arg in args]
        return Code(self.code + "(" + ", ".join(arg.code for arg in args) + ")")

def quote(x): # is quote really the right word? Reify?
    return Code(repr(x))


def quote_lam(f, n):
    names = [f"x{i}" for i in range(n)]
    body = f(*[Code(name) for name in names])
    return Code("lambda " + ", ".join(names) + ": " + body.code)
# I could discover n in an error loop. Cheeky. Or maybe inspect can determine?

def quote_lam(f):
    names = inspect.signature(f).parameters.keys()
    body = f(*[Code(name) for name in names])
    return Code("lambda " + ", ".join(names) + ": " + body.code)
# I could discover n in an error loop. Cheeky. Or maybe inspect can determine?

def quote(x):
    if isinstance(x, Code):
        return Code(repr(x))
    elif callable(x):
        return quote_lam(x)
    else:
        return Code(repr(x))


#quote_lam(lambda x,y: x, 2)
quote_lam(lambda x,y: x)

quote("hello")
quote(lambda x,y: x + 3 + 4)

assert quote(lambda x,y: 3 + 4 + x) == Code("lambda x, y: 7 + x") # a little normalization
assert quote(lambda x : [1,3,4] + [4,5,6] + x) == Code("lambda x: [1, 3, 4, 4, 5, 6] + x")

def pow(x, n):
    if n == 0:
        return 1
    elif n == 1:
        return x
    else:
        return x * pow(x, n - 1)

assert quote(Code("3")) == Code("Code(code='3')")
quote(lambda x: pow(x, 3))
# But the typically problem of not being able to overload if then else rears it's head.

NameError: name 'dataclass' is not defined

In [ ]:
# kind of more de bruijn environmental?
def quote_lam(f):
    argslen = len(inspect.signature(f).parameters)
    body = f(*[Code(f"args[{i}]") for i in range(argslen)])
    return Code("lambda *args: " + body.code)
quote_lam(lambda x,y: x)

Code(code='lambda *args: args[0]')

In [28]:
import inspect
sig = inspect.signature(lambda x,y: x)
sig.return_annotation

for x in sig.parameters:
    print(repr(x))

'x'
'y'


In [ ]:
dir(sig)

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '_bind',
 '_bound_arguments_cls',
 '_hash_basis',
 '_parameter_cls',
 '_parameters',
 '_return_annotation',
 'bind',
 'bind_partial',
 'empty',
 'from_callable',
 'parameters',
 'replace',
 'return_annotation']

In [20]:
sig.parameters

mappingproxy({'x': <Parameter "x">, 'y': <Parameter "y">})

https://docs.python.org/3/library/string.templatelib.html

In [12]:
pi = 3.14
t't-strings are new in Python {pi!s}!'
import sys
sys.version

t't-strings are new in Python {pi!s}!'
import string.templatelib as tl


t"{3} + {4}"
t"{3} + {4}"


def pow(x,n):
    if n == 0:
        return "1"
    elif n == 1:
        return t"{x}"
    else:
        return t"{x} * {pow(x, n - 1)}"
pow("x", 3)


Template(strings=('', ' * ', ''), interpolations=(Interpolation('x', 'x', None, ''), Interpolation(Template(strings=('', ' * ', ''), interpolations=(Interpolation('x', 'x', None, ''), Interpolation(Template(strings=('', ''), interpolations=(Interpolation('x', 'x', None, ''),)), 'pow(x, n - 1)', None, ''))), 'pow(x, n - 1)', None, '')))

version